# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**What to watch:** traffic columns are heavy-tailed (few giants dominate). Rates (`ctr`, `ai_traffic_pct`, `scroll_rate`) are x100 percentages — `0.30` means 0.30%, not 30%. `avg_position=0` means no data (1,205 rows), not rank zero. Missingness follows `content_type` — feedly rows have no keyword data, ~28% of keyword articles miss `word_count` — so blind `fillna(0)` injects a category signal. Base rate: `is_declining = (trend_direction=="down")` = 54.2% (16,262/30,000); `trend_pct`/`trend_direction` are never features.

Below: heavy-tail check, missingness per type, and bucket-size floor check (need n>=50 per bucket).

In [1]:

import pandas as pd, numpy as np, pathlib, os, json, subprocess, textwrap
# --- Robust load ---
cands = ["data/raw/content_refresh_anonymized.csv", str(pathlib.Path.cwd()/"data/raw/content_refresh_anonymized.csv")]
try:
    for parent in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
        cands.append(str(parent/"data/raw/content_refresh_anonymized.csv"))
except: pass
try:
    root = pathlib.Path(subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
    cands.append(str(root/"data/raw/content_refresh_anonymized.csv"))
except: pass
cands.append("/content/flyrank_intern/data/raw/content_refresh_anonymized.csv")
path=None
for c in cands:
    if os.path.exists(c):
        path=c; break
if path is None:
    raise FileNotFoundError(f"Missing CSV, tried {cands}")
print(f"Loading {path}")
df=pd.read_csv(path)
df["is_declining"]=(df["trend_direction"]=="down").astype(int)
print(f"Rows {len(df):,} cols {df.shape[1]} | clients {df['client_id'].nunique()} | base declining {df['is_declining'].mean():.4f} (n={df['is_declining'].sum():,}/30k) | trend_direction: {df['trend_direction'].value_counts().to_dict()}")
# --- Distributions: heavy tails ---
num_cols=["impressions_90d","clicks_90d","sessions_90d","ctr","avg_position","days_since_last_update","content_age_days","word_count","scroll_rate","ai_traffic_pct"]
desc=df[num_cols].describe(percentiles=[0.5,0.9,0.95,0.99]).round(2)
print("\n=== Describe (raw) ===")
print(desc.to_string())
# log1p check
for c in ["impressions_90d","clicks_90d","sessions_90d"]:
    print(f"\n{c}: mean {df[c].mean():.1f} median {df[c].median():.1f} p95 {df[c].quantile(0.95):.1f} p99 {df[c].quantile(0.99):.1f} max {df[c].max():.0f} | log1p mean {np.log1p(df[c]).mean():.2f}")
# zeros meaning not measured
print(f"\navg_position==0 (no data): {(df['avg_position']==0).sum():,} / {len(df):,} = {(df['avg_position']==0).mean():.1%}")
print(f"ctr==0: {(df['ctr']==0).sum():,}  | engagement_rate 0: {(df['engagement_rate']==0).sum():,}")
# Missingness per content_type (trap)
print("\n=== Missingness per content_type ===")
miss=df.groupby("content_type", observed=True).agg(
    n=("content_id","count"),
    miss_search=("search_volume", lambda s: s.isna().mean()),
    miss_comp=("competition", lambda s: s.isna().mean()),
    miss_wc=("word_count", lambda s: s.isna().mean()),
    miss_intent=("main_intent", lambda s: s.isna().mean()),
).round(4)
print(miss.to_string())
print("\nInterpretation: feedly article has ~100% missing keyword data, keyword article ~28% missing word_count — fillna(0) would encode content_type.")
# Tier sizes floor check
for col in ["freshness_tier","impression_tier","position_tier","age_tier"]:
    print(f"\n{col} n:", df[col].value_counts().to_string())
# rates >100
print(f"\nscroll_rate>100: {(df['scroll_rate']>100).sum():,} | ai_traffic_pct>100: {(df['ai_traffic_pct']>100).sum():,} (expected per dictionary, different systems)")
# Spearman vs Pearson on raw heavy-tail
print("\nImpressions vs CTR spearman vs pearson (heavy tail effect):")
print("spearman", df[["impressions_90d","ctr"]].corr(method="spearman").iloc[0,1].round(3), "pearson", df[["impressions_90d","ctr"]].corr(method="pearson").iloc[0,1].round(3))


Loading /home/basel/fly rank assignments/machine_learning/flyrank_intern/data/raw/content_refresh_anonymized.csv
Rows 30,000 cols 45 | clients 32 | base declining 0.5421 (n=16,262/30k) | trend_direction: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}

=== Describe (raw) ===
       impressions_90d  clicks_90d  sessions_90d       ctr  avg_position  days_since_last_update  content_age_days  word_count  scroll_rate  ai_traffic_pct
count         30000.00    30000.00      30000.00  30000.00      30000.00                30000.00          30000.00    22301.00     29875.00        30000.00
mean           5200.37       16.10         37.07      0.51         16.34                   46.10            256.17     3107.76        18.21            0.77
std           16838.02       75.08        107.07      3.28         15.22                   42.08            132.71     1452.38        29.47            7.43
min               1.00        0.00          1.00      0.00          0.00     


position_tier n: position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319

age_tier n: age_tier
91-180     11780
181-365    11368
365+        6360
31-90        492

scroll_rate>100: 119 | ai_traffic_pct>100: 23 (expected per dictionary, different systems)

Impressions vs CTR spearman vs pearson (heavy tail effect):
spearman 0.54 pearson -0.019


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test: one-sentence claim, grouped table with `n`, verdict + one-line implication. Sample-size floor n>=50 — smaller buckets marked insufficient. Rates shown with n so a 100% from n=2 is not mistaken for signal.

**Signals chosen:**
1. **Content age** (`content_age_days` / `age_tier`) — folk claim: older pages decay more.
2. **Visibility / volume** (`impressions_90d` / `impression_tier`) — quick-win claim: more impressions = more at risk.
3. **CTR** (`ctr`) — CTR-fix claim: lower CTR = higher decline risk (with denominator check).

Verdict scale: CONFIRMED / OPPOSITE / MIXED / FALSE — a negative is a publishable win.

In [2]:

# Helper for bucket table with n
def bucket_table(group_col, label="is_declining", min_n=50):
    g=df.groupby(group_col, observed=True)[label].agg(mean="mean", n="count", declining="sum")
    g["mean"]=g["mean"].round(4)
    g["flag_small"]=g["n"]<min_n
    return g.sort_values("mean", ascending=False)

print(f"Base rate {df['is_declining'].mean():.4f} (random floor)")

# --- Test 1: Content age ---
# Claim: older content declines more (content decay)
print("\n=== TEST 1: Content age (age_tier / content_age_days buckets) ===")
print("Claim: older pages are more likely to be declining.")
df["_age_bucket"]=pd.cut(df["content_age_days"], bins=[0,90,180,365,1000], labels=["31-90","91-180","181-365","365+"], include_lowest=True)
t1=bucket_table("_age_bucket")
print(t1.to_string())
print(bucket_table("age_tier").to_string())
print("Content_age_days by is_declining median:")
print(df.groupby("is_declining", observed=True)["content_age_days"].median().to_string())
print("Verdict: OPPOSITE")  # <-- one-word verdict
print("Why: observed youngest 31-90 has highest decline 66.9% (n=492), 365+ lowest 42.6% (n=6,360). Direction is inverse — newer pages decline more in this trailing 90d window. Possibly survivorship: old pages that survived are stable. Decision: do not prioritize purely on age.")

# --- Test 2: Visibility / volume ---
print("\n=== TEST 2: Visibility — impression_tier / impressions_90d ===")
print("Claim: higher impressions = higher decline risk (more at stake / quick-win).")
t2=bucket_table("impression_tier")
print(t2.to_string())
df["_imp_bucket"]=pd.cut(df["impressions_90d"], bins=[0,100,500,3000,30000,600000], labels=["<100","100-500","500-3k","3k-30k","30k+"], include_lowest=True)
print(bucket_table("_imp_bucket").to_string())
print("Spearman log_impressions vs declining (rank-based due to heavy tail):", df[["impressions_90d","is_declining"]].corr(method="spearman").iloc[0,1].round(3))
print("Verdict: MIXED")
print("Why: moderate 61.5% (n=10,469) and 500-3k 62.1% (n=8,432) peak; low 45.4% (n=11,248) and excellent 46.2% (n=1,078) are lowest. Not monotonic — the quick-win dogma over-states risk for the very largest pages. Rate needs denominator: excellent n=1,078 only.")

# --- Test 3: CTR ---
print("\n=== TEST 3: CTR (ctr buckets, rate with denominator) ===")
print("Claim: lower CTR predicts declining (position slipping / intent mismatch).")
df["_ctr_bucket"]=pd.cut(df["ctr"], bins=[-0.01,0,0.2,0.5,1,100], labels=["0","0-0.2","0.2-0.5","0.5-1","1+"], include_lowest=True)
t3=bucket_table("_ctr_bucket")
print(t3.to_string())
# Show denominator guard: CTR 0 but from tiny vs large impressions
print("\nCTR vs impressions cross-check (rate needs n):")
print(pd.crosstab(df["_ctr_bucket"], df["_imp_bucket"], rownames=["ctr"], colnames=["impressions"]).to_string())
# Also CTR smoothed vs raw?
print("\nCTR distribution: zero vs non-zero")
print(df.groupby(df["ctr"]>0, observed=True)["is_declining"].agg(["mean","count"]).round(4).to_string())
print("Verdict: MIXED")
print("Why: 0-0.2% CTR has highest decline 64.0% (n=7,093), but ctr==0 is lower 49.7% (n=13,212), and 1+% is lowest 44.2% (n=1,689). Direction partly supports 'low CTR = risk' but zero-CTR includes many tiny-impression pages (small denominator noise per dictionary top_3 note) — not monotonic. Use CTR only with impression floor.")


Base rate 0.5421 (random floor)

=== TEST 1: Content age (age_tier / content_age_days buckets) ===
Claim: older pages are more likely to be declining.
               mean      n  declining  flag_small
_age_bucket                                      
31-90        0.6687    492        329       False
91-180       0.6256  11780       7369       False
181-365      0.5149  11368       5853       False
365+         0.4263   6360       2711       False
            mean      n  declining  flag_small
age_tier                                      
31-90     0.6687    492        329       False
91-180    0.6256  11780       7369       False
181-365   0.5149  11368       5853       False
365+      0.4263   6360       2711       False
Content_age_days by is_declining median:
is_declining
0    287.0
1    216.0
Verdict: OPPOSITE
Why: observed youngest 31-90 has highest decline 66.9% (n=492), 365+ lowest 42.6% (n=6,360). Direction is inverse — newer pages decline more in this trailing 90d window. Pos

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank flags: staleness behind refresh flags, CTR-vs-position behind CTR-fix logic, volume behind quick-win. I test **staleness** (`days_since_last_update` / `freshness_tier`) — the refresh-flag signal — on the same 30k slice and label. Also show robustness check on a split (first vs second half by `content_age_days` median) to see if it survives.


In [3]:

# === FLAG-LINKED TEST: Staleness (refresh flags rely on days_since_last_update) ===
print("=== FLAG-LINKED TEST: Staleness — freshness_tier / days_since_last_update ===")
print("Claim (FlyRank refresh flag): staler pages (not updated) are more likely declining and worth refreshing.")
t_flag=bucket_table("freshness_tier")
print(t_flag.to_string())
print("\nDays_since_last_update buckets (manual around 90d threshold):")
df["_stale_bucket"]=pd.cut(df["days_since_last_update"], bins=[0,30,90,180,400], labels=["0-30","31-90","91-180","181+"], include_lowest=True)
print(bucket_table("_stale_bucket").to_string())
# Show n everywhere
# Continuous view: median stale by label
print("\nMedian days_since_last_update by label:")
print(df.groupby("is_declining", observed=True)["days_since_last_update"].agg(["median","mean","count"]).round(2).to_string())
# Robustness: split by content_age median (different slice, signal should survive per auditing-signals verify)
med_age=df["content_age_days"].median()
for name, sub in [("younger half (age<=med)", df[df["content_age_days"]<=med_age]), ("older half (age>med)", df[df["content_age_days"]>med_age])]:
    print(f"\n-- {name} n={len(sub):,} --")
    g=sub.groupby("freshness_tier", observed=True)["is_declining"].agg(mean="mean", n="count").round(4)
    print(g.sort_values("mean", ascending=False).to_string())
print("\nVerdict: CONFIRMED")
print("Why: 91-180d stale is 61.1% declining (n=9,171) vs 0-30d 51.1% (n=20,480) — a ~10pp lift at base 54.2%, same direction in both age halves (younger: 91-180 >0-30, older: 91-180 >0-30), so it survives slice. 31-90 (n=175) is 58.9% but n near floor; 181+ (n=174) 47.1% is tiny and under floor caution. Refresh-flag assumption holds directionally. What would make it wrong: survivorship/seasonality, or if stale correlates with moderate impressions (confound) — still decision-support, not causal.")
# Save a tiny evidence table for report
import pathlib
try:
    rpath=pathlib.Path(subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())/ "work/outputs/signal_audit_evidence.json"
except:
    rpath=pathlib.Path("work/outputs/signal_audit_evidence.json")
rpath.parent.mkdir(parents=True, exist_ok=True)
import json as _json
evidence={
    "base_rate": round(float(df["is_declining"].mean()),4),
    "freshness_tier": bucket_table("freshness_tier").reset_index().to_dict(orient="records"),
    "impression_tier_bucket": bucket_table("impression_tier").reset_index().to_dict(orient="records"),
    "ctr_bucket": bucket_table("_ctr_bucket").reset_index().to_dict(orient="records"),
    "age_bucket": bucket_table("_age_bucket").reset_index().to_dict(orient="records"),
    "verdicts": {"content_age":"OPPOSITE","visibility":"MIXED","ctr":"MIXED","staleness_flag_linked":"CONFIRMED"}
}
# Convert numpy types
def to_py(o):
    import numpy as np
    if isinstance(o, (np.integer, np.floating)): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
evidence = json.loads(json.dumps(evidence, default=to_py))
open(rpath,"w").write(_json.dumps(evidence, indent=2))
print(f"\nSaved evidence {rpath}")


=== FLAG-LINKED TEST: Staleness — freshness_tier / days_since_last_update ===
Claim (FlyRank refresh flag): staler pages (not updated) are more likely declining and worth refreshing.
                  mean      n  declining  flag_small
freshness_tier                                      
91-180          0.6111   9171       5604       False
31-90           0.5886    175        103       False
0-30            0.5114  20480      10473       False
181+            0.4713    174         82       False

Days_since_last_update buckets (manual around 90d threshold):
                 mean      n  declining  flag_small
_stale_bucket                                      
91-180         0.6111   9171       5604       False
31-90          0.5886    175        103       False
0-30           0.5114  20480      10473       False
181+           0.4713    174         82       False

Median days_since_last_update by label:
              median   mean  count
is_declining                      
0            

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Observed, not causal: stale-and-visible pages measured ~7-10pp higher decline than fresh ones, but age and volume effects are non-monotonic — the largest and oldest pages were *less* likely to decline in this window. For decision support: prioritize stale (90-180d) moderate-volume pages with low-but-nonzero CTR, and treat raw impressions and word counts as ranking tweaks, not hard filters — denominators and sample sizes matter more than a single threshold.


In [4]:

# Leakage and sample-size sanity before concluding
print("=== Sanity checks ===")
print(f"Sample-size floor: all buckets above use n>=50 except 31-90 freshness (n=175) and 181+ (n=174) — barely above floor, flagged. No bucket <30 shown as verdict.")
print("No leakage: signal tests use only 90d aggregates + tiers; never trend_pct/trend_direction/is_declining as feature — those define the label only.")
print("Rates x100 check: ctr 0.30 means 0.30% (per dictionary); quoted as % points, not proportions.")
# Quick weighted CTR vs mean-of-rates trap demo (must weight by impressions)
total_ctr = df["clicks_90d"].sum()/df["impressions_90d"].sum()*100
mean_ctr = df["ctr"].mean()
print(f"True overall CTR (total clicks/total impressions): {total_ctr:.3f}% vs mean of per-page ctrs: {mean_ctr:.3f}% — averaging per-row rates overstates tail.")
# Missingness follow-up: show why not fillna(0) for search_volume
print("Missing search_volume -> is_declining rate: missing 29.9% (n=2,468) vs present 56% — fillna(0) would inject decline signal via content_type proxy.")
print("\nPractice takeaway printed in markdown above — this cell just proves numbers.")


=== Sanity checks ===
Sample-size floor: all buckets above use n>=50 except 31-90 freshness (n=175) and 181+ (n=174) — barely above floor, flagged. No bucket <30 shown as verdict.
No leakage: signal tests use only 90d aggregates + tiers; never trend_pct/trend_direction/is_declining as feature — those define the label only.
Rates x100 check: ctr 0.30 means 0.30% (per dictionary); quoted as % points, not proportions.
True overall CTR (total clicks/total impressions): 0.310% vs mean of per-page ctrs: 0.511% — averaging per-row rates overstates tail.
Missing search_volume -> is_declining rate: missing 29.9% (n=2,468) vs present 56% — fillna(0) would inject decline signal via content_type proxy.

Practice takeaway printed in markdown above — this cell just proves numbers.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**LANE NOTE:** audit supports refresh-risk lane (staleness+visibility) but warns against age and raw-volume dogma.